# Real Scan Inference
Run a real face scan through backbone → transformer → coarse matching → Sinkhorn → LGR.

**Note:** The decoder (morphing) is intentionally bypassed — it doesn't generalize to real scans yet.
Registration is done against `mean_ref` (avg face), which works correctly.

In [1]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SCAN_FILE    = '2189.npy'   # filename relative to experiment dir
SNAPSHOT     = 'epoch-30.pth.tar'   # checkpoint filename inside output/.../snapshots/
SCALE_FACTOR = 1/1.17  #1.05              # raw_pts /= SCALE_FACTOR to reach UHM meter-scale
APPLY_FLIP   = True  #False              # flip Z axis (True for some legacy scans)
DEVICE       = 'cuda'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SCAN_FILE    = 'planck_1.npy'#'plank_scaled.npy'   # filename relative to experiment dir
SNAPSHOT     = 'epoch-30.pth.tar'   # checkpoint filename inside output/.../snapshots/
SCALE_FACTOR = 1.05              # raw_pts /= SCALE_FACTOR to reach UHM meter-scale
APPLY_FLIP   =  False              # flip Z axis (True for some legacy scans)
DEVICE       = 'cuda'
# ─────────────────────────────────────────────────────────────────────────────

In [13]:
import os, sys

EXP_DIR  = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.dirname(os.path.dirname(EXP_DIR))
sys.path.insert(0, EXP_DIR)
sys.path.insert(0, ROOT_DIR)
os.chdir(EXP_DIR)

import torch
import torch.nn.functional as F
import numpy as np
import plotly.graph_objects as go

from config_dowsampled import make_cfg
from dataset import train_valid_data_loader
from model import create_model

from geotransformer.utils.data import precompute_data_stack_mode
from geotransformer.modules.ops import point_to_node_partition, index_select
from geotransformer.modules.ops.transformation import apply_transform

cfg = make_cfg()
cfg.data.dataset_root = os.path.join(ROOT_DIR, 'data', 'faces')

_, _, neighbor_limits = train_valid_data_loader(cfg, distributed=False)
print('neighbor_limits:', neighbor_limits)

SNAP_DIR = os.path.join(
    ROOT_DIR, 'output',
    'geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn',
    #'6.voxel_subsample',
    'snapshots',
)
ckpt_path = os.path.join(SNAP_DIR, SNAPSHOT)

model = create_model(cfg).to(DEVICE)
model.neighbor_limits = neighbor_limits
ckpt = torch.load(ckpt_path, map_location=DEVICE)
sd   = ckpt.get('model', ckpt)
sd   = {k.replace('module.', ''): v for k, v in sd.items()}

missing = set(model.state_dict()) - set(sd)
extra   = set(sd) - set(model.state_dict())
if missing: print(f'[warn] missing keys (random init): {len(missing)}')
if extra:   print(f'[info] extra keys ignored: {len(extra)}')

model.load_state_dict(sd, strict=False)
model.eval()
print(f'Loaded: {SNAPSHOT}  (epoch {ckpt.get("epoch", "?")})')

with torch.no_grad():
    mean_ref_pts = model.generate_reference_geometry(torch.zeros(32, 100, device=DEVICE))
mean_ref_np = mean_ref_pts.cpu().numpy()
print(f'Mean ref: {mean_ref_pts.shape}')

neighbor_limits: [60 25 29 31]
Loaded: epoch-30.pth.tar  (epoch 30)
Mean ref: torch.Size([10788, 3])


In [14]:
raw = np.load(os.path.join(EXP_DIR, SCAN_FILE))
src_pts = torch.from_numpy(raw[:, :3].astype(np.float32)).to(DEVICE)

src_pts /= SCALE_FACTOR

if APPLY_FLIP:
    src_pts[:, 2] *= -1

src_pts -= src_pts.mean(dim=0, keepdim=True)
src_np   = src_pts.cpu().numpy()

print(f'Scan      : {SCAN_FILE}  ({src_pts.shape[0]} pts)')
print(f'Src range : x=[{src_np[:,0].min():.3f}, {src_np[:,0].max():.3f}]'
      f'  y=[{src_np[:,1].min():.3f}, {src_np[:,1].max():.3f}]'
      f'  z=[{src_np[:,2].min():.3f}, {src_np[:,2].max():.3f}]')

Scan      : planck_1.npy  (2000 pts)
Src range : x=[-0.649, 0.646]  y=[-0.794, 0.785]  z=[-0.430, 0.369]


In [15]:
def pcd_trace(pts, color, name, size=2, opacity=0.7):
    if isinstance(pts, torch.Tensor):
        pts = pts.cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )

def show_reg(traces, title, height=600):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=height,
        scene=dict(aspectmode='data'),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()

def rot_angle_deg(T):
    R = T[:3, :3].cpu().numpy() if isinstance(T, torch.Tensor) else T[:3, :3]
    cos_val = np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_val)))

def nn_dist_m(A, B):
    """Mean nearest-neighbour distance from A to B (metres)."""
    if isinstance(A, np.ndarray):
        A = torch.from_numpy(A).float().to(DEVICE)
    if isinstance(B, np.ndarray):
        B = torch.from_numpy(B).float().to(DEVICE)
    return torch.cdist(A, B).min(dim=1).values.mean().item()

In [16]:
# ── Bypass: backbone → transformer → coarse → Sinkhorn → LGR ─────────────────
# Morphing (decoder) is intentionally skipped — it doesn't generalise to real scans yet.

n_ref = mean_ref_pts.shape[0]
n_src = src_pts.shape[0]

concat_pts  = torch.cat([mean_ref_pts, src_pts], dim=0)
lengths_cpu = torch.tensor([n_ref, n_src], dtype=torch.int64)

graph = precompute_data_stack_mode(
    concat_pts.cpu(), lengths_cpu,
    model.num_stages, model.init_voxel_size, model.init_radius, model.neighbor_limits,
)
for key in ['points', 'lengths', 'neighbors', 'subsampling', 'upsampling']:
    if key in graph:
        graph[key] = [t.to(DEVICE) for t in graph[key]]

dd = {**graph, 'features': torch.ones(n_ref + n_src, 1, device=DEVICE)}

with torch.no_grad():
    feats_list = model.backbone(dd['features'], dd)
    feats_c    = feats_list[-1]
    feats_f    = feats_list[0]

    ref_len_c = dd['lengths'][-1][0].item()
    ref_len_f = dd['lengths'][1][0].item()
    ref_len   = dd['lengths'][0][0].item()

    pts_c     = dd['points'][-1]
    ref_pts_c = pts_c[:ref_len_c];  src_pts_c = pts_c[ref_len_c:]
    pts_f     = dd['points'][1]
    ref_pts_f = pts_f[:ref_len_f];  src_pts_f = pts_f[ref_len_f:]
    pts_0     = dd['points'][0]
    ref_pts_0 = pts_0[:ref_len];    src_pts_0 = pts_0[ref_len:]

    _, ref_node_masks, ref_knn_idx, ref_knn_masks = point_to_node_partition(
        ref_pts_f, ref_pts_c, model.num_points_in_patch
    )
    _, src_node_masks, src_knn_idx, src_knn_masks = point_to_node_partition(
        src_pts_f, src_pts_c, model.num_points_in_patch
    )

    ref_pad_f = torch.cat([ref_pts_f, torch.zeros_like(ref_pts_f[:1])], dim=0)
    src_pad_f = torch.cat([src_pts_f, torch.zeros_like(src_pts_f[:1])], dim=0)
    ref_knn_pts = index_select(ref_pad_f, ref_knn_idx, dim=0)
    src_knn_pts = index_select(src_pad_f, src_knn_idx, dim=0)

    ref_fc_raw  = feats_c[:ref_len_c]
    src_fc_raw  = feats_c[ref_len_c:]
    ref_feats_f = feats_f[:ref_len_f]
    src_feats_f = feats_f[ref_len_f:]

    ref_fc_out, src_fc_out = model.transformer(
        ref_pts_c.unsqueeze(0), src_pts_c.unsqueeze(0),
        ref_fc_raw.unsqueeze(0), src_fc_raw.unsqueeze(0),
    )
    ref_feats_c = F.normalize(ref_fc_out.squeeze(0), p=2, dim=1)
    src_feats_c = F.normalize(src_fc_out.squeeze(0), p=2, dim=1)

    ref_ci, src_ci, corr_scores_c = model.coarse_matching(
        ref_feats_c, src_feats_c, ref_node_masks, src_node_masks
    )

    ref_ck_idx   = ref_knn_idx[ref_ci];    src_ck_idx   = src_knn_idx[src_ci]
    ref_ck_masks = ref_knn_masks[ref_ci];  src_ck_masks = src_knn_masks[src_ci]
    ref_ck_pts   = ref_knn_pts[ref_ci];    src_ck_pts   = src_knn_pts[src_ci]

    ref_pad_ff = torch.cat([ref_feats_f, torch.zeros_like(ref_feats_f[:1])], dim=0)
    src_pad_ff = torch.cat([src_feats_f, torch.zeros_like(src_feats_f[:1])], dim=0)
    ref_ck_feats = index_select(ref_pad_ff, ref_ck_idx, dim=0)
    src_ck_feats = index_select(src_pad_ff, src_ck_idx, dim=0)

    ms = torch.einsum('bnd,bmd->bnm', ref_ck_feats, src_ck_feats) / feats_f.shape[1] ** 0.5
    ms = model.optimal_transport(ms, ref_ck_masks, src_ck_masks)
    if not model.fine_matching.use_dustbin:
        ms = ms[:, :-1, :-1]

    ref_cp, src_cp, cp_scores, est_T = model.fine_matching(
        ref_ck_pts, src_ck_pts, ref_ck_masks, src_ck_masks, ms, corr_scores_c,
    )

src_aligned = apply_transform(src_pts_0, est_T)
angle       = rot_angle_deg(est_T)
t_norm      = float(est_T[:3, 3].norm().item())
nn_d        = nn_dist_m(src_aligned.cpu().numpy(), ref_pts_0.cpu().numpy())

print(f'Scan           : {SCAN_FILE}')
print(f'Checkpoint     : {SNAPSHOT}')
print(f'R angle        : {angle:.2f}°')
print(f't norm         : {t_norm:.4f} m')
print(f'Mean NN dist   : {nn_d:.4f} m')
print(f'R diagonal     : {est_T[0,0]:.3f}  {est_T[1,1]:.3f}  {est_T[2,2]:.3f}')
print(f'Coarse corr    : {ref_ci.shape[0]}')
print(f'Fine corr      : {ref_cp.shape[0]}')
print(f'Ref superpoints: {ref_pts_c.shape[0]}')
print(f'Src superpoints: {src_pts_c.shape[0]}')
print(f'\nEstimated transform:')
print(est_T.cpu().numpy())

Scan           : planck_1.npy
Checkpoint     : epoch-30.pth.tar
R angle        : 133.50°
t norm         : 0.6312 m
Mean NN dist   : 0.2388 m
R diagonal     : -0.684  0.984  -0.677
Coarse corr    : 256
Fine corr      : 3147
Ref superpoints: 395
Src superpoints: 105

Estimated transform:
[[-0.6836379   0.02959089 -0.729221    0.14019546]
 [ 0.14726487  0.9842183  -0.09812099  0.12119471]
 [ 0.71480906 -0.17446786 -0.67720675  0.6034242 ]
 [ 0.          0.          0.          1.        ]]


---
## Visualizations

In [17]:
show_reg(
    [pcd_trace(mean_ref_np, 'steelblue', 'mean ref'),
     pcd_trace(src_np,      'tomato',    f'real scan ({src_pts.shape[0]} pts)')],
    f'{SCAN_FILE} — raw positions (before alignment)',
)

In [18]:
show_reg(
    [pcd_trace(ref_pts_0,   'steelblue', 'mean ref (stage-1)'),
     pcd_trace(src_aligned, 'orange',    f'real scan aligned  R={angle:.1f}°  NN={nn_d:.4f}m')],
    f'{SCAN_FILE} — predicted alignment',
)

In [19]:
# Coarse correspondences
ref_pts_c_np = ref_pts_c.cpu().numpy()
src_pts_c_np = src_pts_c.cpu().numpy()
ref_ci_np    = ref_ci.cpu().numpy()
src_ci_np    = src_ci.cpu().numpy()
scores_c_np  = corr_scores_c.cpu().numpy()

xs, ys, zs = [], [], []
for ri, si in zip(ref_ci_np, src_ci_np):
    xs += [float(ref_pts_c_np[ri, 0]), float(src_pts_c_np[si, 0]), None]
    ys += [float(ref_pts_c_np[ri, 1]), float(src_pts_c_np[si, 1]), None]
    zs += [float(ref_pts_c_np[ri, 2]), float(src_pts_c_np[si, 2]), None]

fig = go.Figure(data=[
    pcd_trace(ref_pts_c_np, 'steelblue', f'ref superpoints ({len(ref_pts_c_np)})', size=5, opacity=0.5),
    pcd_trace(src_pts_c_np, 'orange',    f'src superpoints ({len(src_pts_c_np)})', size=5, opacity=0.5),
    pcd_trace(ref_pts_c_np[ref_ci_np], 'darkblue', f'ref matched ({len(ref_ci_np)})', size=7),
    pcd_trace(src_pts_c_np[src_ci_np], 'crimson',  f'src matched ({len(src_ci_np)})', size=7),
    go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                 line=dict(color='gray', width=1), opacity=0.3, name='correspondences'),
])
fig.update_layout(
    title='Coarse correspondences',
    height=650, scene=dict(aspectmode='data'), margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

In [20]:
# Fine correspondences (pre-transform src)
ref_cp_np = ref_cp.cpu().numpy()
src_cp_np = src_cp.cpu().numpy()
cp_sc_np  = cp_scores.cpu().numpy()

print(f'Fine correspondences: {len(ref_cp_np)} inlier pairs')
print(f'Score range: [{cp_sc_np.min():.4f}, {cp_sc_np.max():.4f}]  mean={cp_sc_np.mean():.4f}')

xs_f, ys_f, zs_f = [], [], []
for rp, sp in zip(ref_cp_np, src_cp_np):
    xs_f += [float(rp[0]), float(sp[0]), None]
    ys_f += [float(rp[1]), float(sp[1]), None]
    zs_f += [float(rp[2]), float(sp[2]), None]

fig_f = go.Figure(data=[
    go.Scatter3d(x=xs_f, y=ys_f, z=zs_f, mode='lines',
                 line=dict(color='gray', width=1), opacity=0.2, name='fine corr lines'),
    go.Scatter3d(
        x=ref_cp_np[:, 0], y=ref_cp_np[:, 1], z=ref_cp_np[:, 2],
        mode='markers',
        marker=dict(size=4, color=cp_sc_np, colorscale='Viridis',
                    colorbar=dict(title='score'), opacity=0.9),
        name='ref corr pts',
    ),
    go.Scatter3d(
        x=src_cp_np[:, 0], y=src_cp_np[:, 1], z=src_cp_np[:, 2],
        mode='markers',
        marker=dict(size=4, color=cp_sc_np, colorscale='Plasma', opacity=0.9),
        name='src corr pts (raw)',
    ),
])
fig_f.update_layout(
    title='Fine correspondences — ref (Viridis) ↔ src (Plasma), color=score',
    height=650, scene=dict(aspectmode='data'), margin=dict(l=0, r=0, b=0, t=40),
)
fig_f.show()

Fine correspondences: 3147 inlier pairs
Score range: [0.0500, 0.5061]  mean=0.1274


---
## Full pipeline pass
Backbone → transformer → coarse → coeff_regressor → morphing → Sinkhorn → LGR.
The estimated transform here may differ from the bypass above if the morphed ref diverges from mean_ref.

In [21]:
n_ref_fp = mean_ref_pts.shape[0]
n_src_fp = src_pts.shape[0]

data_dict_full = {
    'points'   : torch.cat([mean_ref_pts, src_pts], dim=0),  # keep on DEVICE — model derives graph device from this
    'lengths'  : torch.tensor([n_ref_fp, n_src_fp], dtype=torch.int64),
    'features' : torch.ones(n_ref_fp + n_src_fp, 1, device=DEVICE),
    'gt_z'     : torch.zeros(32, 100, device=DEVICE),
    'transform': torch.eye(4, device=DEVICE),
}

with torch.no_grad():
    out_full = model(data_dict_full)

est_T_full   = out_full['estimated_transform']
morphed_np   = out_full['morphed_full'].cpu().numpy()
z_np         = out_full['z_coefficients'].cpu().numpy()  # [32, 100]
src_aln_full = apply_transform(out_full['src_points'], est_T_full)

angle_full  = rot_angle_deg(est_T_full)
t_norm_full = float(est_T_full[:3, 3].norm().item())
nn_d_full   = nn_dist_m(src_aln_full.cpu().numpy(), out_full['ref_points'].cpu().numpy())

print(f'=== Full pipeline ===')
print(f'R angle      : {angle_full:.2f}°')
print(f't norm       : {t_norm_full:.4f} m')
print(f'Mean NN dist : {nn_d_full:.4f} m  (vs morphed ref)')
print(f'R diagonal   : {est_T_full[0,0]:.3f}  {est_T_full[1,1]:.3f}  {est_T_full[2,2]:.3f}')
print()
print(f'=== pred_z stats ===')
print(f'abs mean : {np.abs(z_np).mean():.6f}')
print(f'abs max  : {np.abs(z_np).max():.6f}')
patch_norms = np.linalg.norm(z_np, axis=1)
print(f'per-patch L2 norm:')
for i, n in enumerate(patch_norms):
    print(f'  patch {i:2d}: {n:.6f}')
delta = morphed_np - mean_ref_np
print(f'\nVertex displacement — mean: {np.abs(delta).mean():.6f} m  max: {np.abs(delta).max():.6f} m')
print(f'\nEstimated transform:')
print(est_T_full.cpu().numpy())

show_reg(
    [pcd_trace(mean_ref_np, 'steelblue', 'mean ref'),
     pcd_trace(morphed_np,  'tomato',    'morphed ref (pred z)')],
    f'{SCAN_FILE} — mean ref vs morphed ref',
)
show_reg(
    [pcd_trace(out_full['ref_points'].cpu().numpy(), 'steelblue', 'morphed ref'),
     pcd_trace(src_aln_full.cpu().numpy(),           'orange',    f'src aligned (full pipeline)  R={angle_full:.1f}°')],
    f'{SCAN_FILE} — full pipeline alignment',
)

print(f'\n=== Bypass vs full pipeline ===')
print(f'Bypass  : R={angle:.2f}°  t={t_norm:.4f}m  NN={nn_d:.4f}m')
print(f'Full    : R={angle_full:.2f}°  t={t_norm_full:.4f}m  NN={nn_d_full:.4f}m')

=== Full pipeline ===
R angle      : 130.24°
t norm       : 0.6079 m
Mean NN dist : 0.2242 m  (vs morphed ref)
R diagonal   : -0.645  0.996  -0.643

=== pred_z stats ===
abs mean : 0.000288
abs max  : 0.004107
per-patch L2 norm:
  patch  0: 0.005261
  patch  1: 0.005197
  patch  2: 0.005150
  patch  3: 0.005198
  patch  4: 0.005197
  patch  5: 0.005147
  patch  6: 0.005162
  patch  7: 0.005186
  patch  8: 0.005044
  patch  9: 0.005147
  patch 10: 0.005120
  patch 11: 0.005092
  patch 12: 0.005201
  patch 13: 0.005181
  patch 14: 0.005294
  patch 15: 0.005252
  patch 16: 0.005231
  patch 17: 0.005192
  patch 18: 0.005257
  patch 19: 0.005199
  patch 20: 0.005319
  patch 21: 0.005149
  patch 22: 0.005044
  patch 23: 0.005107
  patch 24: 0.005107
  patch 25: 0.005160
  patch 26: 0.005177
  patch 27: 0.005158
  patch 28: 0.005326
  patch 29: 0.005182
  patch 30: 0.005086
  patch 31: 0.005196

Vertex displacement — mean: 0.000086 m  max: 0.000728 m

Estimated transform:
[[-0.64479065  0.013


=== Bypass vs full pipeline ===
Bypass  : R=133.50°  t=0.6312m  NN=0.2388m
Full    : R=130.24°  t=0.6079m  NN=0.2242m
